In [2]:
import cv2
import time
from ultralytics import YOLO

In [3]:
ruta_video = "video.mp4"
ruta_salida = "salida_video.mp4"

confianza = 0.40
iou_nms = 0.60

# YOLOv8n: liviano y rápido para un MVP
modelo = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(ruta_video)

if not cap.isOpened():
    print("No se pudo abrir el video.")
    exit()

In [4]:
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_video = cap.get(cv2.CAP_PROP_FPS)

if fps_video <= 0:
    fps_video = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(ruta_salida, fourcc, fps_video, (ancho, alto))

nombres_clases = modelo.names

while True:
    inicio = time.time()

    ret, frame = cap.read()
    if not ret:
        break

    resultados = modelo(frame, conf=confianza, iou=iou_nms, verbose=False)

    for resultado in resultados:
        cajas = resultado.boxes

        for caja in cajas:
            x1, y1, x2, y2 = caja.xyxy[0].tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            clase_id = int(caja.cls[0].item())
            confianza_det = float(caja.conf[0].item())
            etiqueta = f"{nombres_clases[clase_id]} {confianza_det:.2f}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame,
                etiqueta,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    fin = time.time()
    fps_inferencia = 1 / (fin - inicio) if (fin - inicio) > 0 else 0

    cv2.putText(
        frame,
        f"FPS: {fps_inferencia:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    out.write(frame)
    cv2.imshow("Deteccion YOLO en video", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Proceso terminado.")
print(f"Video guardado en: {ruta_salida}")

Proceso terminado.
Video guardado en: salida_video.mp4
